# Inner London cyclability：阶段 1 数据盘点与裁剪

本 notebook 完成四件事：

1. 用行政区边界圈定 Inner London，并外扩缓冲区
2. 从 Code-Point 生成起点（邮编质心）
3. 把道路、绿地、POI 裁到研究范围，并检查字段
4. 保存为 `work/stage1.gpkg`，供后续 isochrone 和指标计算使用

使用方式：只改第 2 节的 `INNER` 和 `CONFIG`，然后从头运行到底。把第 5 节的打印结果发回即可。

## 1. 环境与常量

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from pyogrio import list_layers, read_info

CRS = "EPSG:27700"
BUFFER_M = 3000   # 研究区外扩距离，避免边缘 isochrone 被截断
CELL_M = 100      # 仅用于统计：按该栅格去重后还剩多少个起点

OUT = Path("work/stage1.gpkg")
OUT.parent.mkdir(parents=True, exist_ok=True)

## 2. 配置

`INNER` 默认是法定 Inner London（12 个区加 City of London，含 Greenwich）。若采用其他口径，直接改名单。名称按"包含"匹配，所以 `Hammersmith` 能匹配 `Hammersmith and Fulham`。

In [ ]:
INNER = [
    "City of London", "Camden", "Greenwich", "Hackney", "Hammersmith",
    "Islington", "Kensington", "Lambeth", "Lewisham", "Southwark",
    "Tower Hamlets", "Wandsworth", "Westminster",
]

CONFIG = {
    # 区级边界多边形（Boundary-Line 或 ONS）；name_col 为区名所在列
    "boundary":   {"path": "data/boundary_lad.gpkg", "layer": None, "name_col": "NAME"},
    # Code-Point CSV，需含 Eastings / Northings
    "codepoint":  {"path": "data/codepoint.csv"},
    "roads":      {"path": "data/oproad.gpkg", "layer": "road_link"},
    "greenspace": {"path": "data/opgrsp.gpkg", "layer": "greenspace_site"},
    "poi":        {"path": "data/poi.gpkg", "layer": None},
}

## 3. 辅助函数

In [ ]:
def read_layer(path, layer=None, bounds=None):
    """读取空间图层。仅当文件本身是 EPSG:27700 时才用 bbox 预过滤，读完统一转 27700。"""
    path = str(path)
    if layer is None:
        layers = list_layers(path)
        print(f"  {path} 内的图层: {[l[0] for l in layers]}")
        layer = layers[0][0]
    info = read_info(path, layer=layer)
    in_27700 = bool(info.get("crs")) and "27700" in str(info["crs"])
    bbox = tuple(bounds) if (bounds is not None and in_27700) else None
    if bounds is not None and not in_27700:
        print(f"  注意: {path} 不是 EPSG:27700 ({info.get('crs')})，将读取整个文件")
    gdf = gpd.read_file(path, layer=layer, bbox=bbox)
    if gdf.crs is None:
        raise ValueError(f"{path}: 未定义坐标系")
    return gdf.to_crs(CRS)


def select_intersecting(gdf, geom):
    """保留与 geom 相交的要素（整条保留，不切断）。"""
    idx = gdf.sindex.query(geom, predicate="intersects")
    return gdf.iloc[sorted(idx)].copy()


def report(name, gdf, show_counts=()):
    """打印行数、字段、几何类型，以及指定字段的取值分布。"""
    print(f"== {name}: {len(gdf):,} 行, crs={gdf.crs}")
    print("字段:", [c for c in gdf.columns if c != "geometry"])
    print("几何类型:", gdf.geom_type.value_counts().to_dict())
    for col in show_counts:
        if col in gdf.columns:
            print(f"\n-- {col}")
            print(gdf[col].value_counts(dropna=False).head(30).to_string())
        else:
            print(f"\n-- 没有字段 {col}")
    if not show_counts:
        print(gdf.drop(columns="geometry").head(3).T.to_string())

## 4. 研究区

In [ ]:
b = CONFIG["boundary"]
bnd = read_layer(b["path"], b["layer"])
print("边界文件中的名称（前 60 个）:")
print(sorted(bnd[b["name_col"]].astype(str).unique())[:60])

In [ ]:
mask = bnd[b["name_col"]].astype(str).apply(
    lambda s: any(k.lower() in s.lower() for k in INNER)
)
matched = bnd[mask]
print(f"匹配到 {len(matched)} 个，名单要求 {len(INNER)} 个:")
print(sorted(matched[b["name_col"]].astype(str)))
if len(matched) != len(INNER):
    print("\n请检查：匹配数与 INNER 名单不一致，先调整名称再继续")

study = matched.dissolve()
study_geom = study.geometry.iloc[0]
study_buf = study_geom.buffer(BUFFER_M)
bounds = study_buf.bounds
print(f"\n研究区面积: {study_geom.area / 1e6:.1f} km²")

## 5. 起点（Code-Point 邮编质心）

坐标为 0 的邮编（定位质量标识 90，即无坐标）会被剔除。若 CSV 没有表头，按 Code-Point Open 的 10 列顺序补列名。

In [ ]:
cp_path = CONFIG["codepoint"]["path"]
cp = pd.read_csv(cp_path, low_memory=False)
cp.columns = [str(c).upper() for c in cp.columns]

if "EASTINGS" not in cp.columns:  # 无表头的 Code-Point Open
    cp = pd.read_csv(
        cp_path, header=None, low_memory=False,
        names=["POSTCODE", "POSITIONAL_QUALITY_INDICATOR", "EASTINGS", "NORTHINGS",
               "COUNTRY_CODE", "NHS_REGIONAL_HA_CODE", "NHS_HA_CODE",
               "ADMIN_COUNTY_CODE", "ADMIN_DISTRICT_CODE", "ADMIN_WARD_CODE"],
    )

cp = cp[(cp["EASTINGS"] > 0) & (cp["NORTHINGS"] > 0)]
origins = gpd.GeoDataFrame(
    cp, geometry=gpd.points_from_xy(cp["EASTINGS"], cp["NORTHINGS"]), crs=CRS
)
origins = origins[origins.within(study_geom)].reset_index(drop=True)

cell_id = (origins.geometry.x // CELL_M).astype(int).astype(str) + "_" + \
          (origins.geometry.y // CELL_M).astype(int).astype(str)
print(f"Inner London 内的起点: {len(origins):,} 个邮编")
print(f"按 {CELL_M} m 栅格去重后: {cell_id.nunique():,} 个")

## 6. 裁剪各图层并检查字段

### 6.1 道路（OS Open Roads）

重点看 `road_classification`：A Road、B Road 为主干道，Classified Unnumbered 对应 C，Unclassified 对应 U，Motorway 需剔除。

In [ ]:
c = CONFIG["roads"]
roads = select_intersecting(read_layer(c["path"], c["layer"], bounds), study_buf)
report("roads", roads, show_counts=("road_classification", "road_function", "form_of_road"))

### 6.2 绿地（OS Open Greenspace）

In [ ]:
c = CONFIG["greenspace"]
green = select_intersecting(read_layer(c["path"], c["layer"], bounds), study_buf)
report("greenspace", green, show_counts=("function",))

### 6.3 兴趣点（POI）

字段因数据版本而异，这里只打印前三行，用来确定服务类型对应哪一列。

In [ ]:
c = CONFIG["poi"]
poi = select_intersecting(read_layer(c["path"], c["layer"], bounds), study_buf)
report("poi", poi)

## 7. 目视检查

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
roads.plot(ax=ax, linewidth=0.2, color="0.6")
green.plot(ax=ax, color="tab:green", alpha=0.4)
origins.sample(min(len(origins), 20000), random_state=0).plot(ax=ax, markersize=0.3, color="tab:red")
study.boundary.plot(ax=ax, color="black", linewidth=1)
gpd.GeoSeries([study_buf], crs=CRS).boundary.plot(ax=ax, color="black", linestyle="--", linewidth=0.8)
ax.set_title("Study area (solid) and buffer (dashed)")
ax.set_axis_off()
plt.show()

## 8. 保存

In [ ]:
layers_out = {
    "study_area": study,
    "study_buffer": gpd.GeoDataFrame(geometry=[study_buf], crs=CRS),
    "origins": origins,
    "roads": roads,
    "greenspace": green,
    "poi": poi,
}
OUT.unlink(missing_ok=True)
for name, g in layers_out.items():
    g.to_file(OUT, layer=name, driver="GPKG")
print(f"已保存 {list(layers_out)} 到 {OUT}")